###🎯 What you will learn:

✅  Learn how Overlay (intersection) works in GeoPandas

✅  Clip roads by governorate boundaries

✅  Calculate total road length per governorate

In [ ]:
# Import Libraries
import geopandas as gpd
import matplotlib.pyplot as plt
import os

In [ ]:
# mount google drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Define output folder directory
output_folder = '/content/drive/MyDrive/2-GeoPandas Mini Course/4-Overlay/Output'

In [ ]:
# Datasets Paths
# administrative boundaries (ADM1) Shapefile
adm_path = '/content/drive/MyDrive/2-GeoPandas Mini Course/2-GeoPackage Data Handling, CRS & Cleaning/Data/egy_admin1.shp'

# Egypt OSM data GeoPackage
osm_path = '/content/drive/MyDrive/2-GeoPandas Mini Course/2-GeoPackage Data Handling, CRS & Cleaning/Data/egypt.gpkg'


###Load the datasets

In [ ]:
# Load ADM1 Data
adm = gpd.read_file(adm_path)
adm.head()

In [ ]:
# Keep only needed columns
adm = adm[['adm1_name', 'geometry']].copy()
adm

In [ ]:
# Inspect GeoPackage Layers
osm_layers = gpd.list_layers(osm_path)
osm_layers

In [ ]:
# Load POIS Point Data
roads = gpd.read_file(osm_path, layer='gis_osm_roads_free')
roads

In [ ]:
# Keep only necessary columns
cols = ['fclass', 'name', 'geometry']
roads_sub = roads[cols].copy()

roads_sub

In [ ]:
# Check all unique vlaues in fclass
roads_sub.fclass.unique()

In [ ]:
# Filter Primary Roads
primary_roads = roads_sub[roads_sub['fclass'] == 'primary']
primary_roads

###Align CRS (projected for length)

In [ ]:
# Convert to projected UTM Zone CRS (meters)
adm = adm.to_crs(epsg=32636)
primary_roads = primary_roads.to_crs(epsg=32636)

###Overlay (intersection)

In [ ]:
# Clip roads by governorate boundaries
roads_clipped = gpd.overlay(primary_roads, adm, how='intersection')
roads_clipped

In [ ]:
# Calculate road length in kilometers
roads_clipped['length_km'] = roads_clipped.length / 1000
roads_clipped

In [ ]:
# Total road length per governorate
roads_summary = (
    roads_clipped
    .groupby('adm1_name')['length_km']
    .sum()
    .reset_index() # Convert Series to DataFrame
    .sort_values('length_km', ascending=False) # Sort by 'length_km' column
    .head(10) # Get top 10
    .reset_index(drop=True) # Reset index and drop the old one
)

roads_summary

###Bar Plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Explicitly tell pandas which columns to use
roads_summary.plot(
    kind='bar',
    x='adm1_name',    # This puts the names on the X-axis
    y='length_km',    # This puts the values on the Y-axis
    ax=ax,
    color='teal',
    edgecolor='black'
)

plt.title('Top 10 Governorates by Primary Road Length', fontsize=14, pad=15)
plt.xlabel('Governorate', fontsize=12)
plt.ylabel('Total Length (km)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Save the figure as a high-resolution PNG
#fig.savefig(os.path.join(output_folder, 'Top 10 Governorates by Primary Road Length.png'), dpi=300, bbox_inches='tight')   # Uncomment to use